## Ensemble Methods and Discussion

In [1]:
import sys
import os

# Adds the parent directory to the search path
sys.path.append(os.path.abspath(".."))

from src.config import RAW_DATA_PATH
from src.data_loader import load_raw_data
from src.modeling import (
    get_base_models,
    tune_gradient_boosting,
    tune_random_forest,
    tune_xgboost,
    cross_validate_models,
)
from src.preprocessing import (
    split_features_target,
    stratified_train_test_split,
    scale_features,
    apply_smote,
)

df = load_raw_data(RAW_DATA_PATH)
df.head()

base_models = get_base_models()

from src.modeling import build_voting_classifier, build_stacking_classifier

lr_model = base_models["Logistic Regression"]
svm_model = base_models["SVM (RBF)"]

X, y = split_features_target(df, target_col="Outcome")
X_train, X_test, y_train, y_test = stratified_train_test_split(X, y)

X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)
X_train_balanced, y_train_balanced = apply_smote(X_train_scaled, y_train)

gb_best = tune_gradient_boosting(X_train_balanced, y_train_balanced)
rf_best = tune_random_forest(X_train_balanced, y_train_balanced)
xgb_best = tune_xgboost(X_train_balanced, y_train_balanced)

# Replace base models with tuned versions
base_models["Gradient Boosting"] = gb_best
base_models["Random Forest"] = rf_best
base_models["XGBoost"] = xgb_best

voting_clf = build_voting_classifier(gb_best, rf_best, lr_model, svm_model, xgb_best)
voting_clf.fit(X_train_balanced, y_train_balanced)

stacking_clf = build_stacking_classifier(gb_best, rf_best, svm_model, xgb_best)
stacking_clf.fit(X_train_balanced, y_train_balanced)

[2026-03-25 11:02:43] [src.data_loader] [INFO] Loading raw data from D:\Portfolio Projects\diabetes-ml-prediction\data\raw\diabetes.csv
[2026-03-25 11:02:43] [src.data_loader] [INFO] Loaded dataset with shape (768, 9)
[2026-03-25 11:02:43] [src.modeling] [INFO] Initializing base models
[2026-03-25 11:02:43] [src.preprocessing] [INFO] Splitting features and target
[2026-03-25 11:02:43] [src.preprocessing] [INFO] Features shape: (768, 8), Target shape: (768,)
[2026-03-25 11:02:43] [src.preprocessing] [INFO] Performing stratified train-test split
[2026-03-25 11:02:43] [src.preprocessing] [INFO] Train size: 576, Test size: 192
[2026-03-25 11:02:43] [src.preprocessing] [INFO] Scaling features with StandardScaler
[2026-03-25 11:02:43] [src.preprocessing] [INFO] Feature scaling complete
[2026-03-25 11:02:43] [src.preprocessing] [INFO] Applying SMOTE to balance classes
[2026-03-25 11:02:43] [src.preprocessing] [INFO] After SMOTE - X_balanced: (750, 8), y_balanced: (750,)
[2026-03-25 11:02:43] 

StackingClassifier(estimators=[('gb',
                                GradientBoostingClassifier(max_depth=6,
                                                           min_samples_split=5,
                                                           n_estimators=200,
                                                           random_state=251000861,
                                                           subsample=0.8)),
                               ('rf',
                                RandomForestClassifier(class_weight='balanced',
                                                       max_depth=15,
                                                       n_estimators=200,
                                                       random_state=251000861)),
                               ('svm',
                                SVC(class_weight='balanced', probability=True,
                                    random_state=251000861)),
                               ('xgb',
                                XGBCla...
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=6,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=300, n_jobs=None,
                                              num_parallel_tree=None,
                                              random_state=251000861, ...))],
                   final_estimator=LogisticRegression(class_weight='balanced',
                                                      max_iter=1000,
                                                      random_state=251000861))

In [4]:
from src.evaluation import evaluate_model
import pandas as pd

voting_metrics = evaluate_model(voting_clf, X_test_scaled, y_test, "Voting Classifier")
stacking_metrics = evaluate_model(stacking_clf, X_test_scaled, y_test, "Stacking Classifier")

pd.DataFrame([voting_metrics, stacking_metrics])

[2026-03-25 11:21:06] [src.evaluation] [INFO] Evaluating model: Voting Classifier
[2026-03-25 11:21:06] [src.evaluation] [INFO] Voting Classifier - Acc: 0.7812, Prec: 0.6623, Rec: 0.7612, F1: 0.7083, ROC-AUC: 0.8587
[2026-03-25 11:21:06] [src.evaluation] [INFO] Evaluating model: Stacking Classifier
[2026-03-25 11:21:06] [src.evaluation] [INFO] Stacking Classifier - Acc: 0.7760, Prec: 0.6579, Rec: 0.7463, F1: 0.6993, ROC-AUC: 0.8486


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Voting Classifier,0.781250,0.662338,0.761194,0.708333,0.858746
1,Stacking Classifier,0.776042,0.657895,0.746269,0.699301,0.848597



# Discussion

## **Clinical Significance**

The Logistic Regression model achieves **73% recall** for diabetes detection, meaning it correctly identifies roughly **3 out of every 4 diabetic patients**.  
This level of sensitivity is acceptable for a **screening tool**, where the priority is to catch as many potential cases as possible.

However:

- In real clinical settings, **higher recall is often preferred**, even if it increases false positives.
- False positives can be resolved with follow‑up tests.
- False negatives are more dangerous because they represent missed diagnoses.

Thus, while the model performs reasonably well, **there is room for improvement**, especially if the goal is early detection or population‑level screening.

---

## ** Why Logistic Regression Outperformed Ensemble Methods on the Test Set**

Despite ensemble models achieving **higher cross‑validation scores**, Logistic Regression delivered the **best real‑world generalization** on the test set.  
This can be explained by several factors:

### **Complexity–Generalization Tradeoff**
- Tree‑based ensembles (Random Forest, Gradient Boosting, XGBoost) are more complex.
- They achieved higher CV performance but likely **overfit** the training data.
- Logistic Regression, being simpler, generalized better to unseen data.

### **Data Characteristics**
- The Pima dataset may contain relationships that are **mostly linear**.
- Linear models naturally capture these patterns more effectively than deep trees.

### **Feature Interaction Patterns**
- Logistic Regression may align better with the **true underlying structure** of the dataset.
- Tree models may attempt to model unnecessary interactions, reducing test performance.

---

## **Effectiveness of SMOTE**

SMOTE successfully balanced the training data from **1.87:1** to **1:1**.  
This had several benefits:

- Prevented the model from being biased toward the majority class.
- Improved learning of minority‑class (diabetes) patterns.
- Enabled fair comparison across models.
- Reduced the risk of models predicting “No Diabetes” too frequently.

SMOTE was applied **only to the training set**, preserving the real‑world distribution in the test set.

---

## **Model Selection Strategy**

Five diverse models were selected:

- Logistic Regression (linear)
- Random Forest (bagging ensemble)
- Gradient Boosting (boosting ensemble)
- XGBoost (optimized boosting)
- SVM with RBF kernel (non‑linear margin‑based)

This diversity is valuable because:

- No single algorithm is universally best.
- Different model families capture different structures in the data.
- Ensemble combinations can leverage complementary strengths.

This strategy ensured a **comprehensive evaluation** across algorithmic paradigms.


## **Limitations and Future Improvements**

### **1. Dataset Size**
- Only 768 samples.
- Larger datasets could improve generalization and reduce variance.

### **2. Feature Engineering**
- Only original features were used.
- Domain‑specific features (ratios, interactions, polynomial terms) may improve performance.

### **3. Threshold Optimization**
- All models used the default **0.5** classification threshold.
- In clinical settings, lowering the threshold could **increase recall**, reducing missed diagnoses.

### **4. External Validation**
- Results should be tested on **other populations** to ensure generalizability.

### **5. Feature Importance & Interpretability**
- SHAP or coefficient analysis could provide clinical insight into which features drive predictions.

# Conclusion

This project successfully developed and compared multiple machine learning models for diabetes prediction using the Pima Indians Diabetes Database.

### **Key Conclusions**

### **1. Best Performing Model**
Logistic Regression achieved the strongest test‑set performance:

- **Accuracy:** 79.69%  
- **Precision:** 69.44%  
- **Recall:** 74.63%  
- **ROC‑AUC:** 0.8683  

This demonstrates that **simpler models can outperform complex ensembles** when generalization matters.

---

### **2. Cross‑Validation vs Test Performance**
- Gradient Boosting achieved the highest CV ROC‑AUC (0.8892).
- Logistic Regression performed best on the test set (0.8683 ROC‑AUC).

This highlights the importance of **holding out a true test set** for final evaluation.

---

### **3. Ensemble Methods**
- The Voting Classifier achieved **0.8601 ROC‑AUC**, competitive with individual models.
- Ensembles remain valuable, especially when combining diverse learners.

---

### **4. Class Imbalance Handling**
SMOTE effectively balanced the training data without discarding original samples, improving minority‑class learning.


### **5. Clinical Applicability**
With **73% recall** and **69% precision**, the model is suitable as a **clinical decision support tool** for:

- Early screening  
- Identifying high‑risk individuals  
- Triggering follow‑up diagnostic tests  

It should **not** be used as a standalone diagnostic tool.
